In [62]:
import pandas as pd
from pathlib import Path
from sklearn.metrics import roc_auc_score
import warnings
warnings.filterwarnings("ignore")

In [50]:
data_dir = Path("../data/raw")

In [63]:
clicks_train = pd.read_csv(data_dir / "clicks_train.csv.zip")
events = pd.read_csv(data_dir / "events.csv.zip")
promoted_content = pd.read_csv(data_dir / "promoted_content.csv.zip")
documents_categories = pd.read_csv(data_dir / "documents_categories.csv.zip")

In [52]:
sample_display_ids = clicks_train["display_id"].drop_duplicates().sample(1_000_000, random_state=42)
clicks_sample = clicks_train[clicks_train["display_id"].isin(sample_display_ids)]

In [53]:
merged = clicks_sample.merge(events, on="display_id").merge(
    promoted_content, on="ad_id", suffixes=("_view", "_ad")
)
merged.columns.tolist()

['display_id',
 'ad_id',
 'clicked',
 'uuid',
 'document_id_view',
 'timestamp',
 'platform',
 'geo_location',
 'document_id_ad',
 'campaign_id',
 'advertiser_id']

In [54]:
val_display_ids = merged["display_id"].drop_duplicates().sample(frac=0.2, random_state=42)
val = merged[merged["display_id"].isin(val_display_ids)]
train = merged[~merged["display_id"].isin(val_display_ids)]

train.shape, val.shape

((4130698, 11), (1032724, 11))

In [55]:
train.columns.tolist()

['display_id',
 'ad_id',
 'clicked',
 'uuid',
 'document_id_view',
 'timestamp',
 'platform',
 'geo_location',
 'document_id_ad',
 'campaign_id',
 'advertiser_id']

In [56]:
ad_ctr = train.groupby("ad_id")["clicked"].mean()
global_ctr = train["clicked"].mean()

In [57]:
val = val.copy()
val["score_popularity"] = val["ad_id"].map(ad_ctr).fillna(global_ctr)

In [58]:
primary_category = (
    documents_categories.sort_values("confidence_level", ascending=False)
    .drop_duplicates("document_id")
    .set_index("document_id")["category_id"]
)

train = train.copy()
train["category_id"] = train["document_id_ad"].map(primary_category)
val["category_id"] = val["document_id_ad"].map(primary_category)

category_ctr = train.groupby("category_id")["clicked"].mean()
val["score_category"] = val["category_id"].map(category_ctr).fillna(global_ctr)

In [59]:
def mrr_at_display(df, score_col):
    sorted_df = df.sort_values(["display_id", score_col], ascending=[True, False]).copy()
    sorted_df["rank"] = sorted_df.groupby("display_id").cumcount() + 1
    reciprocal_rank = 1 / sorted_df.loc[sorted_df["clicked"] == 1, "rank"]
    return reciprocal_rank.mean()

In [60]:
roc_auc_score(val["clicked"], val["score_popularity"]), mrr_at_display(val, "score_popularity")

(0.7041838098337504, 0.6342580115440115)

In [61]:
roc_auc_score(val["clicked"], val["score_category"]), mrr_at_display(val, "score_category")

(0.5795035531130364, 0.5351013924963925)